# Project 1 — Data Quality Check
## Supply Chain Inventory & Demand Planning Analytics

This notebook profiles the raw supply-chain datasets, validates key data-quality and business rules, and creates clean processed datasets for downstream analysis.

**Workflow:** Raw Data → Profiling → Validation → Cleaning → Processed Data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data: {RAW_DIR}')
print(f'Processed data: {PROCESSED_DIR}')

## 1. Load Raw Datasets

In [ ]:
suppliers = pd.read_csv(RAW_DIR / 'suppliers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
inventory = pd.read_csv(RAW_DIR / 'inventory.csv')
sales = pd.read_csv(RAW_DIR / 'sales.csv')
purchase_orders = pd.read_csv(RAW_DIR / 'purchase_orders.csv')

datasets = {
    'Suppliers': suppliers,
    'Products': products,
    'Inventory': inventory,
    'Sales': sales,
    'Purchase Orders': purchase_orders
}

for name, df in datasets.items():
    print(f'{name}: {df.shape}')

## 2. Inspect Columns and Data Types

In [ ]:
for name, df in datasets.items():
    print(f'\n{name}')
    print(df.dtypes)

## 3. Missing-Value Assessment

In [ ]:
missing_summary = pd.DataFrame({
    name: df.isna().sum()
    for name, df in datasets.items()
}).fillna(0).astype(int)

display(missing_summary)

## 4. Duplicate-Row Assessment

In [ ]:
duplicate_summary = pd.Series({
    name: int(df.duplicated().sum())
    for name, df in datasets.items()
}, name='duplicate_rows')

display(duplicate_summary.to_frame())

## 5. Primary-Key Validation

In [ ]:
pk_checks = {
    'Supplier ID unique': suppliers['supplier_id'].is_unique,
    'Product ID unique': products['product_id'].is_unique,
    'Sales Order ID unique': sales['order_id'].is_unique,
    'Purchase Order ID unique': purchase_orders['po_id'].is_unique,
    'Inventory composite key unique': not inventory.duplicated(
        subset=['date', 'product_id', 'warehouse_id']
    ).any()
}

for check, result in pk_checks.items():
    print(f'{check}: {result}')

## 6. Inventory Business-Rule Validation

In [ ]:
inventory_numeric = [
    'opening_stock',
    'received_quantity',
    'issued_quantity',
    'closing_stock'
]

negative_inventory = {
    col: int((inventory[col] < 0).sum())
    for col in inventory_numeric
}

inventory['calculated_closing_stock'] = (
    inventory['opening_stock']
    + inventory['received_quantity']
    - inventory['issued_quantity']
)

reconciliation_errors = int(
    (inventory['calculated_closing_stock'] != inventory['closing_stock']).sum()
)

print('Negative inventory values:', negative_inventory)
print(f'Closing-stock reconciliation errors: {reconciliation_errors}')
print(f'Valid inventory records: {len(inventory) - reconciliation_errors:,} / {len(inventory):,}')

## 7. Sales Business-Rule Validation

In [ ]:
sales['unfulfilled_quantity'] = (
    sales['quantity_ordered'] - sales['quantity_fulfilled']
)

negative_sales = {
    'quantity_ordered': int((sales['quantity_ordered'] < 0).sum()),
    'quantity_fulfilled': int((sales['quantity_fulfilled'] < 0).sum())
}

fulfilled_exceeds_ordered = int(
    (sales['quantity_fulfilled'] > sales['quantity_ordered']).sum()
)

print('Negative sales quantities:', negative_sales)
print(f'Fulfilled > ordered errors: {fulfilled_exceeds_ordered}')
print(f'Total ordered: {sales["quantity_ordered"].sum():,}')
print(f'Total fulfilled: {sales["quantity_fulfilled"].sum():,}')
print(f'Total unfulfilled: {sales["unfulfilled_quantity"].sum():,}')

## 8. Purchase-Order Date and Lead-Time Validation

In [ ]:
date_columns = ['order_date', 'expected_date', 'actual_date']

for col in date_columns:
    purchase_orders[col] = pd.to_datetime(purchase_orders[col], errors='coerce')

invalid_dates = {
    col: int(purchase_orders[col].isna().sum())
    for col in date_columns
}

purchase_orders['calculated_lead_time_days'] = (
    purchase_orders['actual_date'] - purchase_orders['order_date']
).dt.days

negative_lead_times = int(
    (purchase_orders['calculated_lead_time_days'] < 0).sum()
)

print('Invalid dates:', invalid_dates)
print(f'Negative lead times: {negative_lead_times}')
print('Lead-time statistics:')
print(purchase_orders['calculated_lead_time_days'].describe())

## 9. Stockout Validation

In [ ]:
stockout_records = int((inventory['closing_stock'] == 0).sum())
nonzero_stockout_records = int(
    ((inventory['closing_stock'] == 0) & (inventory['closing_stock'] != 0)).sum()
)

print(f'Stockout records: {stockout_records:,}')
print(f'Invalid stockout records: {nonzero_stockout_records:,}')

## 10. Create Clean Processed Datasets

In [ ]:
# Reload raw files so temporary validation columns are not saved.
suppliers_clean = pd.read_csv(RAW_DIR / 'suppliers.csv')
products_clean = pd.read_csv(RAW_DIR / 'products.csv')
inventory_clean = pd.read_csv(RAW_DIR / 'inventory.csv')
sales_clean = pd.read_csv(RAW_DIR / 'sales.csv')
purchase_orders_clean = pd.read_csv(RAW_DIR / 'purchase_orders.csv')

for col in ['date']:
    inventory_clean[col] = pd.to_datetime(inventory_clean[col])

sales_clean['order_date'] = pd.to_datetime(sales_clean['order_date'])

for col in ['order_date', 'expected_date', 'actual_date']:
    purchase_orders_clean[col] = pd.to_datetime(purchase_orders_clean[col])

clean_datasets = {
    'suppliers_clean.csv': suppliers_clean,
    'products_clean.csv': products_clean,
    'inventory_clean.csv': inventory_clean,
    'sales_clean.csv': sales_clean,
    'purchase_orders_clean.csv': purchase_orders_clean
}

for filename, df in clean_datasets.items():
    df.to_csv(PROCESSED_DIR / filename, index=False)
    print(f'Saved: {filename} ({len(df):,} rows)')

## 11. Final Data-Quality Summary

In [ ]:
summary = []

for filename, df in clean_datasets.items():
    summary.append({
        'dataset': filename,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_values': int(df.isna().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum())
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

## Conclusion

The raw supply-chain datasets have been profiled and validated against key structural and business rules. Clean versions have been saved to `data/processed/` while the original raw datasets remain unchanged.

The processed datasets are now ready for exploratory data analysis and subsequent inventory, supplier, and demand-planning analysis.